# 13｜刪除站點 ID 缺失旅次，會改變哪些結論？

沿用 06 冊的口徑，以 2025 年 1 月借出的旅次為母體。檔案時間沒有時區偏移，[官方資料說明](https://capitalbikeshare.com/system-data)也未標示時區，本冊直接使用檔案日期及小時。時段合計含一月所有日型，假日照常計入。固定早段為 07:00–09:59，晚段為 16:00–18:59，另比較全部 24 小時的分布。

有站點 ID 代表欄位非空。兩端分別計算一月事件的 ID 涵蓋率，並維持原始站點歸屬。時間與旅次品質檢查沿用 01 冊口徑。

In [1]:
import sqlite3
from zipfile import ZipFile

import pandas as pd
from IPython.display import display

with ZipFile('../data/raw/202501-capitalbikeshare-tripdata.zip') as archive:
    with archive.open('202501-capitalbikeshare-tripdata.csv') as csv:
        rides = pd.read_csv(csv, dtype='string')

started = pd.to_datetime(rides.started_at, format='mixed', errors='coerce')
ended = pd.to_datetime(rides.ended_at, format='mixed', errors='coerce')
assert rides.shape == (285_731, 13)
assert rides.ride_id.fillna('').str.strip().ne('').all()
assert rides.ride_id.is_unique
assert (ended - started).gt(pd.Timedelta(0)).all()

has_start = rides.start_station_id.fillna('').str.strip().ne('')
has_end = rides.end_station_id.fillna('').str.strip().ne('')
jan_start = started.ge('2025-01-01') & started.lt('2025-02-01')
jan_end = ended.ge('2025-01-01') & ended.lt('2025-02-01')

group = pd.Series('兩端皆缺', index=rides.index)
group.loc[~has_start & has_end] = '只缺起站'
group.loc[has_start & ~has_end] = '只缺迄站'
group.loc[has_start & has_end] = '兩端完整'
assert (~has_start).sum() == 88_122
assert (~has_end).sum() == 91_217
assert jan_start.sum() == 285_695
assert jan_end.sum() == 285_731

print(f'借出時間 {started.min()} 至 {started.max()}')
print(f'歸還時間 {ended.min()} 至 {ended.max()}')
missingness = pd.DataFrame({'全檔': group.value_counts(),
                            '一月借出': group.loc[jan_start].value_counts()})
missingness['一月借出占比 %'] = 100 * missingness['一月借出'] / jan_start.sum()
assert missingness['全檔'].sum() == len(rides)
assert missingness['一月借出'].sum() == jan_start.sum()
display(missingness.round(2))

coverage = pd.DataFrame(
    {'一月事件': [jan_start.sum(), jan_end.sum()],
     '有站點 ID': [(jan_start & has_start).sum(), (jan_end & has_end).sum()]},
    index=['借出', '歸還'])
coverage['涵蓋率 %'] = 100 * coverage['有站點 ID'] / coverage['一月事件']
display(coverage.round(2))

借出時間 2024-12-31 19:50:05.679000 至 2025-01-31 23:59:45.752000
歸還時間 2025-01-01 00:00:37.974000 至 2025-01-31 23:59:59.223000


,全檔,一月借出,一月借出占比 %
兩端完整,166773,166757,58.37
兩端皆缺,60381,60373,21.13
只缺迄站,30836,30828,10.79
只缺起站,27741,27737,9.71


,一月事件,有站點 ID,涵蓋率 %
借出,285695,197585,69.16
歸還,285731,194514,68.08


每張比較表以該欄的旅次總數為分母。日期與小時以借出時間計算。差值為「兩端完整 − 全部一月借出」，單位為百分點。圖中的色條共用 0% 至 100% 的比例尺。

In [2]:
events = pd.DataFrame({
    'station_group': group.loc[jan_start],
    'member': rides.member_casual.loc[jan_start].fillna('unknown'),
    'bike': rides.rideable_type.loc[jan_start].fillna('unknown'),
    'date': started.loc[jan_start].dt.strftime('%Y-%m-%d'),
    'hour': started.loc[jan_start].dt.hour,
})
events['period'] = '其他'
events.loc[events.hour.between(7, 9), 'period'] = '早段'
events.loc[events.hour.between(16, 18), 'period'] = '晚段'

totals = events.station_group.value_counts()
totals['全部一月借出'] = len(events)
assert totals['兩端完整'] == 166_757

with sqlite3.connect(':memory:') as db:
    events.to_sql('events', db, index=False)

    for dimension in ('member', 'bike', 'period', 'hour', 'date'):
        counts = pd.read_sql_query(f'''
            SELECT {dimension} AS category, station_group, COUNT(*) AS trips
            FROM events
            GROUP BY {dimension}, station_group
        ''', db).pivot(index='category', columns='station_group', values='trips')
        counts = counts.fillna(0).astype(int)
        counts['全部一月借出'] = counts.sum(axis=1)
        assert counts.sum().eq(totals).all()

        shares = 100 * counts.div(totals, axis='columns')
        comparison = shares[['全部一月借出', '兩端完整']].copy()
        comparison['差值（百分點）'] = comparison['兩端完整'] - comparison['全部一月借出']
        print(dimension)
        display(counts)
        if dimension in ('member', 'bike', 'period'):
            display(comparison.style.format('{:.2f}').bar(
                subset=['全部一月借出', '兩端完整'],
                color='#76b9c5', vmin=0, vmax=100))
        else:
            display(shares.assign(差值百分點=comparison['差值（百分點）']).round(2))

member
bike
period
hour
date


station_group,兩端完整,兩端皆缺,只缺起站,只缺迄站,全部一月借出
category,,,,,
casual,40934,6775,6840,5715,60264
member,125823,53598,20897,25113,225431


station_group,全部一月借出,兩端完整,差值（百分點）
category,,,
casual,21.09,24.55,3.45
member,78.91,75.45,-3.45


station_group,兩端完整,兩端皆缺,只缺起站,只缺迄站,全部一月借出
category,,,,,
classic_bike,78756,0,0,186,78942
electric_bike,88001,60373,27737,30642,206753


station_group,全部一月借出,兩端完整,差值（百分點）
category,,,
classic_bike,27.63,47.23,19.60
electric_bike,72.37,52.77,-19.60


station_group,兩端完整,兩端皆缺,只缺起站,只缺迄站,全部一月借出
category,,,,,
其他,92145,38458,15813,19152,165568
早段,29273,6683,5124,4107,45187
晚段,45339,15232,6800,7569,74940


station_group,全部一月借出,兩端完整,差值（百分點）
category,,,
其他,57.95,55.26,-2.70
早段,15.82,17.55,1.74
晚段,26.23,27.19,0.96


station_group,兩端完整,兩端皆缺,只缺起站,只缺迄站,全部一月借出
category,,,,,
0,1613,941,326,591,3471
1,1120,551,216,273,2160
2,694,340,129,161,1324
3,318,248,79,124,769
4,349,195,99,68,711
5,1273,492,317,223,2305
6,3017,834,638,455,4944
7,7187,1509,1211,850,10757
8,13016,2993,2287,1792,20088


station_group,全部一月借出,兩端完整,兩端皆缺,只缺起站,只缺迄站,差值百分點
category,,,,,,
0,1.21,0.97,1.56,1.18,1.92,-0.25
1,0.76,0.67,0.91,0.78,0.89,-0.08
2,0.46,0.42,0.56,0.47,0.52,-0.05
3,0.27,0.19,0.41,0.28,0.40,-0.08
4,0.25,0.21,0.32,0.36,0.22,-0.04
5,0.81,0.76,0.81,1.14,0.72,-0.04
6,1.73,1.81,1.38,2.30,1.48,0.08
7,3.77,4.31,2.50,4.37,2.76,0.54
8,7.03,7.81,4.96,8.25,5.81,0.77


station_group,兩端完整,兩端皆缺,只缺起站,只缺迄站,全部一月借出
category,,,,,
2025-01-01,4949,2260,862,949,9020
2025-01-02,6417,2542,1005,1083,11047
2025-01-03,5265,2120,921,1028,9334
2025-01-04,4617,1868,807,873,8165
2025-01-05,4900,1842,836,846,8424
2025-01-06,657,816,132,245,1850
2025-01-07,1517,1162,336,424,3439
2025-01-08,2962,1600,595,683,5840
2025-01-09,2984,1779,657,781,6201


station_group,全部一月借出,兩端完整,兩端皆缺,只缺起站,只缺迄站,差值百分點
category,,,,,,
2025-01-01,3.16,2.97,3.74,3.11,3.08,-0.19
2025-01-02,3.87,3.85,4.21,3.62,3.51,-0.02
2025-01-03,3.27,3.16,3.51,3.32,3.33,-0.11
2025-01-04,2.86,2.77,3.09,2.91,2.83,-0.09
2025-01-05,2.95,2.94,3.05,3.01,2.74,-0.01
2025-01-06,0.65,0.39,1.35,0.48,0.79,-0.25
2025-01-07,1.20,0.91,1.92,1.21,1.38,-0.29
2025-01-08,2.04,1.78,2.65,2.15,2.22,-0.27
2025-01-09,2.17,1.79,2.95,2.37,2.53,-0.38


只保留兩端完整旅次會留下 166,757 / 285,695 筆（58.37%）。電輔車占比下降約 19.60 個百分點，會員旅次占比下降約 3.45 個百分點。兩組借出量最高的小時皆為 17:00–17:59，但早段占比提高、晚段占比也提高，因此最高小時一致只代表時段排序的一部分相同。這些是檔案內旅次組成的變化。站點借出和歸還各用該端有 ID 的事件，能保留另一端缺 ID 的旅次。